In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['proyecto_bigdata']
coleccion = db['datos_scraping'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [19]:
import os
import re
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# =========================
# LIMPIEZA
# =========================
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
print("Limpieza lista.")

# =========================
# CONFIGURACIÓN
# =========================
NOMBRE_GRUPO = "AutoTec"
USUARIO = "jocelyn"
MAX_AUTOS = 100

urls_marcas = [
    "https://autos.cari.cl/autos/hyundai",
    "https://autos.cari.cl/autos/chevrolet",
    "https://autos.cari.cl/autos/toyota",
    "https://autos.cari.cl/autos/kia",
    "https://autos.cari.cl/autos/nissan",
    "https://autos.cari.cl/autos/mazda",
    "https://autos.cari.cl/autos/suzuki",
    "https://autos.cari.cl/autos/ford",
    "https://autos.cari.cl/autos/peugeot",
    "https://autos.cari.cl/autos/volkswagen"
]

marcas_validas = [
    "hyundai", "chevrolet", "toyota", "kia", "nissan",
    "mazda", "suzuki", "ford", "peugeot", "volkswagen"
]

datos_finales = []
vistos = set()

# =========================
# SELENIUM
# =========================
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
)

driver = webdriver.Chrome(options=options)

# =========================
# FUNCIONES
# =========================
def limpiar_precio(texto):
    if not texto:
        return 0.0
    limpio = re.sub(r"[^\d]", "", str(texto))
    return float(limpio) if limpio else 0.0

def extraer_precio(texto):
    match = re.search(r"\$\s?[\d\.]+", texto)
    return limpiar_precio(match.group(0)) if match else 0.0

def extraer_anio(texto):
    match = re.search(r"Año\s*(19\d{2}|20\d{2})", texto, re.IGNORECASE)
    if match:
        return int(match.group(1))

    match = re.search(r"\b(19\d{2}|20\d{2})\b", texto)
    return int(match.group(0)) if match else None

def extraer_kilometraje(texto):
    match = re.search(r"(\d[\d\.\,]*)\s*(km|kms|kilómetros|kilometros)", texto.lower())
    if match:
        limpio = re.sub(r"[^\d]", "", match.group(0))
        return int(limpio) if limpio else None
    return None

def extraer_combustible(texto):
    t = texto.lower()
    if "gasolina" in t or "bencina" in t:
        return "Gasolina"
    if "diesel" in t or "diésel" in t:
        return "Diesel"
    if "híbrido" in t or "hibrido" in t:
        return "Híbrido"
    if "eléctrico" in t or "electrico" in t:
        return "Eléctrico"
    return None

def extraer_ciudad(texto):
    ciudades = [
        "Santiago", "La Serena", "Coquimbo", "Valparaíso", "Viña del Mar",
        "Concepción", "Rancagua", "Talca", "Temuco", "Antofagasta",
        "Iquique", "Copiapó", "Chillán", "Puerto Montt", "Osorno",
        "La Florida", "Las Condes", "Maipú", "Providencia"
    ]
    for ciudad in ciudades:
        if ciudad.lower() in texto.lower():
            return ciudad
    return None

def extraer_nombre_auto(lineas):
    for linea in lineas:
        linea_limpia = linea.strip()
        linea_lower = linea_limpia.lower()

        if any(marca in linea_lower for marca in marcas_validas):
            if "$" not in linea_limpia and "año" not in linea_lower and "usado" not in linea_lower:
                return linea_limpia

    return None

def separar_marca_modelo(nombre, marca_url):
    if not nombre:
        return marca_url.capitalize(), None

    partes = nombre.split()
    marca = partes[0]
    modelo = " ".join(partes[1:]) if len(partes) > 1 else None
    return marca, modelo

# =========================
# SCRAPING DESDE BLOQUES
# =========================
try:
    for url in urls_marcas:
        if len(datos_finales) >= MAX_AUTOS:
            break

        marca_url = url.rstrip("/").split("/")[-1]

        print(f"\nProcesando marca: {marca_url}")
        print(url)

        try:
            driver.get(url)
        except Exception as e:
            print("No se pudo abrir la página, se salta:", e)
            continue

        time.sleep(8)

        bloques = driver.find_elements(
            By.CSS_SELECTOR,
            "article, li[itemtype='http://schema.org/Product'], div[class*='item']"
        )

        print("Bloques encontrados:", len(bloques))

        capturados_marca = 0

        for bloque in bloques:
            if len(datos_finales) >= MAX_AUTOS:
                break

            texto = bloque.text.strip()

            if not texto:
                continue

            lineas = [x.strip() for x in texto.split("\n") if x.strip()]
            if not lineas:
                continue

            nombre = extraer_nombre_auto(lineas)

            if not nombre:
                continue

            precio = extraer_precio(texto)

            if precio == 0:
                continue

            marca, modelo = separar_marca_modelo(nombre, marca_url)
            anio = extraer_anio(texto)
            kilometraje = extraer_kilometraje(texto)
            combustible = extraer_combustible(texto)
            ciudad = extraer_ciudad(texto)

            clave = f"{nombre}-{precio}-{anio}"

            if clave in vistos:
                continue

            vistos.add(clave)

            registro = {
                "marca": marca,
                "modelo": modelo,
                "año": anio,
                "kilometraje": kilometraje,
                "combustible": combustible,
                "ciudad": ciudad,
                "url": url,
                "precio": precio,
                "usuario": "jocelyn",
                "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                "grupo": "AutoTec",
                "fuente": "autos.cari.cl"
            }

            datos_finales.append(registro)
            capturados_marca += 1

        print("Capturados en marca:", capturados_marca)
        print("Total acumulado:", len(datos_finales))

    print("\nExtracción terminada:", len(datos_finales), "autos.")

except Exception as e:
    print("Error general:", e)

finally:
    driver.quit()

# =========================
# GUARDAR EN MONGODB LOCAL
# =========================
try:
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    client.admin.command("ping")

    db = client["proyecto_bigdata"]
    coleccion = db["datos_scraping"]

    if datos_finales:
        coleccion.insert_many(datos_finales)
        print(f"Datos cargados en MongoDB correctamente: {len(datos_finales)} registros.")
    else:
        print("No hay datos para guardar.")

except Exception as e:
    print("Error en MongoDB:", e)

# =========================
# VERIFICACIÓN
# =========================
print("Total capturado:", len(datos_finales))

if datos_finales:
    print("Primer registro:")
    print(datos_finales[0])

Limpieza lista.

Procesando marca: hyundai
https://autos.cari.cl/autos/hyundai
Bloques encontrados: 123
Capturados en marca: 18
Total acumulado: 18

Procesando marca: chevrolet
https://autos.cari.cl/autos/chevrolet
Bloques encontrados: 123
Capturados en marca: 20
Total acumulado: 38

Procesando marca: toyota
https://autos.cari.cl/autos/toyota
Bloques encontrados: 123
Capturados en marca: 11
Total acumulado: 49

Procesando marca: kia
https://autos.cari.cl/autos/kia
Bloques encontrados: 123
Capturados en marca: 8
Total acumulado: 57

Procesando marca: nissan
https://autos.cari.cl/autos/nissan
Bloques encontrados: 122
Capturados en marca: 20
Total acumulado: 77

Procesando marca: mazda
https://autos.cari.cl/autos/mazda
Bloques encontrados: 123
Capturados en marca: 20
Total acumulado: 97

Procesando marca: suzuki
https://autos.cari.cl/autos/suzuki
Bloques encontrados: 122
Capturados en marca: 3
Total acumulado: 100

Extracción terminada: 100 autos.
Datos cargados en MongoDB correctamente: 